# Predicción de retrasos en vuelos comerciales
Grupo 11: Mikel Lorite, Adrián Izquierdo y Jon Alba

## Contexto y objetivos del proyecto
El contexto de este proyecto se enmarca en la industria de la aviación comercial, un sector que genera cantidades masivas de datos diariamente y donde la puntualidad operativa es un factor crítico. Para llevar a cabo este estudio, utilizamos el conjunto de datos "2015 Flight Delays and Cancellations", publicado originalmente por la Oficina de Estadísticas de Transporte (Bureau of Transportation Statistics) del Departamento de Transporte de los Estados Unidos (DOT). Este dataset rastrea el rendimiento y la puntualidad de los vuelos domésticos operados por las grandes aerolíneas comerciales dentro del país durante el año 2015.

El objetivo principal de nuestro trabajo es construir un modelo predictivo escalable utilizando Apache Spark que permita anticipar si un vuelo sufrirá un retraso significativo a su llegada. A través de este análisis, buscamos identificar patrones subyacentes en las demoras, respondiendo a preguntas sobre qué factores —como la aerolínea, las infraestructuras de origen/destino o las franjas horarias— inciden en la probabilidad de que un vuelo no cumpla con su horario programado. Adicionalmente, desde la perspectiva de la Ingeniería de Datos, el proyecto tiene como meta evaluar la eficiencia computacional de diversos algoritmos de clasificación binaria (Regresión Logística, Random Forest y Gradient-Boosted Trees) frente a un escenario de alto volumen de datos, midiendo y documentando empíricamente la escalabilidad del clúster a través de pruebas de speed-up y size-up.

## Descripción de los datos 
El conjunto de datos seleccionado representa un volumen de información de gran magnitud, ideal para el procesamiento distribuido. En su totalidad, consta de más de 5,8 millones de registros (5.819.079 observaciones empíricas) estructurados originalmente en 31 variables. Para garantizar la integridad relacional de la información, el dataset se divide en tres archivos en formato CSV independientes pero interconectados:

* flights.csv: Constituye el núcleo central de la información, recogiendo el registro individual de cada vuelo operado durante el año. Entre sus variables métricas y categóricas se incluye información temporal exhaustiva (fecha, horarios de salida y llegada programados frente a los reales) e indicadores de rendimiento operativo. La variable clave para nuestro problema de clasificación subyace en la columna de retraso a la llegada (Arrival Delay), que nos indica en minutos la diferencia respecto al horario previsto.

* airlines.csv: Funciona como una tabla de dimensión o diccionario. Contiene el mapeo directo entre los identificadores de código IATA y el nombre comercial estandarizado de cada compañía aérea, lo que resulta fundamental para la posterior visualización e interpretación de los modelos.

* airports.csv: Actúa como un catálogo geográfico detallado. Este archivo permite enriquecer los datos de los vuelos enlazando los códigos IATA de los aeropuertos de origen y destino con su ubicación física. Proporciona variables demográficas y geoespaciales clave como la ciudad, el estado, la latitud y la longitud, abriendo la puerta a capturar el impacto de la congestión regional en nuestras predicciones.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os


spark = SparkSession.builder.appName("FlightDelays").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 17:35:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/06 17:35:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
import pyspark.sql.functions as F

df_flights = spark.read.csv("data/flights.csv", header=True, inferSchema=True)
df_airlines = spark.read.csv("data/airlines.csv", header=True, inferSchema=True)
df_airports = spark.read.csv("data/airports.csv", header=True, inferSchema=True)

print(f"Total de vuelos iniciales: {df_flights.count():,}")

26/05/06 17:35:17 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


Total de vuelos iniciales: 5,819,079


In [3]:
df_joined = df_flights.join(
    F.broadcast(df_airlines), 
    df_flights.AIRLINE == df_airlines.IATA_CODE, 
    "left"
).drop("IATA_CODE") 

df_airports_orig = df_airports.select(
    F.col("IATA_CODE").alias("ORIGIN_IATA"),
    F.col("LATITUDE").alias("ORIGIN_LAT"),
    F.col("LONGITUDE").alias("ORIGIN_LONG")
)

df_joined = df_joined.join(
    F.broadcast(df_airports_orig), 
    df_joined.ORIGIN_AIRPORT == df_airports_orig.ORIGIN_IATA, 
    "left"
).drop("ORIGIN_IATA")

# Puedes repetir este último paso para DESTINATION_AIRPORT si lo consideras útil

# Explicación de Variables

El archivo `flights.csv` de este conjunto de datos contiene 31 variables que detallan el rendimiento, los tiempos y las causas de retrasos de los vuelos comerciales. Para facilitar su comprensión, aquí tienes las variables resumidas y divididas en categorías lógicas:

### 1. Identificación y Fechas
*   **YEAR, MONTH, DAY:** Año, mes y día en el que se realizó el vuelo. *(Unidad: Numérico)*
*   **DAY_OF_WEEK:** Día de la semana del vuelo. *(Unidad: Numérico, del 1=Lunes al 7=Domingo)*
*   **AIRLINE:** Aerolínea que opera el vuelo. *(Unidad: Texto, código IATA de 2 letras, ej. "AA" para American Airlines)*
*   **FLIGHT_NUMBER:** Número identificador del vuelo. *(Unidad: Numérico)*
*   **TAIL_NUMBER:** Número de matrícula (registro) del avión específico. *(Unidad: Texto)*
*   **ORIGIN_AIRPORT / DESTINATION_AIRPORT:** Aeropuerto de origen y destino final. *(Unidad: Texto, código IATA de 3 letras, ej. "LAX")*

### 2. Tiempos y Horarios (Programados vs. Reales)
*Nota: Todas las variables de esta sección utilizan como unidad el **formato de hora militar (HHMM)** (ej. 1430 = 2:30 PM).*
*   **SCHEDULED_DEPARTURE:** Hora a la que el avión debía salir de la puerta de embarque según el itinerario.
*   **DEPARTURE_TIME:** Hora real en la que el avión se desconectó y empezó a moverse desde la puerta de embarque.
*   **WHEELS_OFF:** Hora exacta en la que el avión despegó y sus ruedas dejaron de tocar la pista.
*   **WHEELS_ON:** Hora exacta en la que las ruedas del avión tocaron la pista de aterrizaje en el destino.
*   **SCHEDULED_ARRIVAL:** Hora a la que el avión debía llegar a su puerta de embarque final.
*   **ARRIVAL_TIME:** Hora real en la que el avión llegó y se estacionó en la puerta de destino.



### 3. Duración y Distancia
*   **TAXI_OUT:** Tiempo transcurrido desde que el avión deja la puerta de embarque hasta que logra despegar (`WHEELS_OFF`). *(Unidad: Minutos)*
*   **AIR_TIME:** Tiempo real que el avión pasó volando (en el aire). *(Unidad: Minutos)*
*   **TAXI_IN:** Tiempo transcurrido desde que el avión aterriza (`WHEELS_ON`) hasta que se estaciona en la puerta de embarque. *(Unidad: Minutos)*
*   **SCHEDULED_TIME:** Duración total del viaje (de puerta a puerta) planeada originalmente. *(Unidad: Minutos)*
*   **ELAPSED_TIME:** Duración total real del viaje (equivale a sumar `TAXI_OUT` + `AIR_TIME` + `TAXI_IN`). *(Unidad: Minutos)*
*   **DISTANCE:** Distancia geográfica recorrida entre los aeropuertos. *(Unidad: Millas)*

### 4. Retrasos y Cancelaciones
*   **DEPARTURE_DELAY:** Diferencia entre la salida real y la programada. *(Unidad: Minutos; un número negativo indica que salió antes de tiempo).*
*   **ARRIVAL_DELAY:** Diferencia entre la llegada real y la programada. *(Unidad: Minutos; un número negativo indica que llegó adelantado).*
*   **DIVERTED:** Indica si el vuelo tuvo que ser desviado a un aeropuerto diferente al planeado. *(Unidad: Binario; 1 = Sí, 0 = No)*
*   **CANCELLED:** Indica si el vuelo fue cancelado por completo. *(Unidad: Binario; 1 = Sí, 0 = No)*
*   **CANCELLATION_REASON:** Motivo de la cancelación del vuelo. *(Unidad: Letra; A = Aerolínea, B = Clima, C = Sistema de Tráfico Aéreo, D = Seguridad)*

**Desglose de Retrasos (Causas específicas)**
*Estas columnas solo registran valores si el retraso total del vuelo superó los 15 minutos.*
*   **AIR_SYSTEM_DELAY:** Retrasos controlados por el sistema nacional de tráfico aéreo (ej. mucho tráfico de vuelos). *(Unidad: Minutos)*
*   **SECURITY_DELAY:** Retrasos debido a brechas de seguridad, evacuaciones o demoras en la inspección. *(Unidad: Minutos)*
*   **AIRLINE_DELAY:** Retraso bajo el control de la aerolínea (mantenimiento del avión, falta de tripulación, retraso en el equipaje, etc.). *(Unidad: Minutos)*
*   **LATE_AIRCRAFT_DELAY:** Efecto dominó; retraso causado porque el avión llegó tarde desde su vuelo previo y demoró la siguiente salida. *(Unidad: Minutos)*
*   **WEATHER_DELAY:** Retrasos causados exclusivamente por condiciones climáticas extremas que impiden volar de forma segura. *(Unidad: Minutos)*

In [4]:
# Filtrar cancelados y desviados
df_clean = df_joined.filter((F.col("CANCELLED") == 0) & (F.col("DIVERTED") == 0))

# Contar nulos en las columnas clave
df_clean.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in ["ARRIVAL_DELAY", "DEPARTURE_DELAY"]]).show()

+-------------+---------------+
|ARRIVAL_DELAY|DEPARTURE_DELAY|
+-------------+---------------+
|            0|              0|
+-------------+---------------+



Tras la ejecución del proceso de limpieza, los reusltado obtenidos confirman la viabilidad del dataset para el entrenamiento de modelos.
El filtrado de vuelos cancelados y desviados ha eliminado los valores faltantes. Como se observa en la salida anterior, ambas variables críticas para la predicción (ARRIVAL_DELAY y DEPARTURE_DELAY) presentan 0 valores nulos. Al disponer de un conjunto de datos limpio con más de 5 millones de registros, garantizamos que las métricas size-up y speed-up que analizaremos más adelante sean representativas del redimiento real.  

In [5]:
df_clean.select("ARRIVAL_DELAY", "DEPARTURE_DELAY", "DISTANCE").summary().show()

26/05/06 17:44:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+-----------------+-----------------+
|summary|    ARRIVAL_DELAY|  DEPARTURE_DELAY|         DISTANCE|
+-------+-----------------+-----------------+-----------------+
|  count|          5714008|          5714008|          5714008|
|   mean|4.407057357987598| 9.29484190431655|824.4569032804994|
| stddev|39.27129709388608|36.88972372075696|608.6619895866803|
|    min|              -87|              -82|               31|
|    25%|              -13|               -5|              373|
|    50%|               -5|               -2|              650|
|    75%|                8|                7|             1065|
|    max|             1971|             1988|             4983|
+-------+-----------------+-----------------+-----------------+



Tras obtener el resumen estadístico de las variables clave podemos extraer las siguientes conclusiones:
- La mediana de ARRIVAL_DELAY es de -5 minutos, lo que significa que más de la mitad de los vuelos llegan antes de su horario programado. Sin embargo, la media es de 4.4 minutos. Esta discrepancia confirma una asimetría positiva en la distribución, causada por una "cola larga" de vuelos con retrasos significativos.
- La desciación típica de los retrasos es casi diez veces superior a la media. Esto indica que mientras que la mayoría de los vuelos son puntuales, los casos de retraso son bastante extremos.
- Se observan valores máximos de hasta 1971 minutos (aproximadamente 32h). Estos outliers son críticos, representan casis que el modelo debe aprender a diferenciar de las fluctuaciones normales de 10 o 15 minutos.
- Existe una diferencia de magnitudes masiva entre las variables. Mientras que los retrasos se mueven en un rango de decenas, las variable DISTANCE alcanza valores de 4983 millas. Esto es lo que justifica la necesidad de aplicar una normalización, evitando que la distancia domine artificialmente los cálculos.
- Dado que el percentil 75% de los retrasos se sitúa en 8 minutos, usaremos el estándar de la FAA (Federal Aviation Administration o Administración Federal de Aviación) que son a partir de los 15 minutos, cuando se considera que un vuelo está retrasado, ya que es cuando empieza a causar problemas como por ejemplo: los pasajeros pierdan sus conexiones, la tripulación puede exceder sus horas legales de trabajo...  Esto nos permitirá centrar la capacidad predictiva del sistema en los retrasos que realmente generan un impacto logístico y económico negativo.

De acuerdo a nuestro objetivo del trabajo, debemos crear una variable que nos especifique si el vuelo se ha retrasado o no, ya que nuestras dos variables binarias del csv nos reflejan si están desviados o calcelados. Para ello usaremos el criterio que hemos mencionado antes.

In [6]:
df_model = df_clean.withColumn("LABEL", F.when(F.col("ARRIVAL_DELAY") >= 15, 1).otherwise(0))

df_model.groupBy("LABEL").count().show()

+-----+-------+
|LABEL|  count|
+-----+-------+
|    1|1063439|
|    0|4650569|
+-----+-------+



In [ ]:
print(df_model.filter((F.col("ARRIVAL_DELAY") < 15) & (F.col("DEPARTURE_DELAY") >= 15)).count())
print(df_model.filter((F.col("ARRIVAL_DELAY") >= 15) & (F.col("DEPARTURE_DELAY") >= 15)).count())

219684


830738


de los que salen tarde, hay 219.684 que llegan bien y 830.738 que llegan tarde tambien

El dataser presenta un desbalanceo de clases moderado proporcionando un 18.6% para los vuelos con retraso y un 81.4% para los vuelos puntuales. El volumen de casos positivos es lo suficiente como para que los algoritmos de Spark aprendan los aptrones del retraso sin necesidad de aplicar técnicaas de sobremuestreo (oversampling).

In [15]:
df_model = df_model.drop("CANCELLED", "DIVERTED")

In [ ]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# 1. Seleccionamos las variables numéricas candidatas
numeric_cols = [
    "SCHEDULED_DEPARTURE", "DEPARTURE_TIME", 
    "DEPARTURE_DELAY", "TAXI_OUT", "WHEELS_OFF", "SCHEDULED_TIME", "ELAPSED_TIME", "AIR_TIME", 
    "DISTANCE", "WHEELS_ON", "TAXI_IN", "SCHEDULED_ARRIVAL", "ARRIVAL_TIME", "ARRIVAL_DELAY", "TARGET"
]

# 2. Preparamos el vector para Spark
assembler = VectorAssembler(inputCols=numeric_cols, outputCol="corr_features")
df_vector = assembler.transform(df_clean).select("corr_features")

# 3. Calculamos la matriz de Pearson
matrix = Correlation.corr(df_vector, "corr_features").collect()[0][0]
cor_np = matrix.toArray()

# 4. Lo pasamos a un DataFrame de Pandas
corr_matrix_df = pd.DataFrame(cor_np, columns=numeric_cols, index=numeric_cols)

# --- NUEVO CÓDIGO: MAPA DE CALOR TRIANGULAR ---

# 5. Generar una máscara para el triángulo superior
# Esto crea una matriz de booleanos donde el triángulo superior es True (y por tanto, se ocultará)
mask = np.triu(np.ones_like(corr_matrix_df, dtype=bool))

# 6. Configurar el tamaño de la figura
plt.figure(figsize=(18, 14))

# 7. Crear el mapa de colores personalizado: de azul claro ('#ADD8E6') a negro ('#000000')
cmap = LinearSegmentedColormap.from_list("light_blue_to_black", ["#ADD8E6", "#000000"])

# 8. Dibujar el mapa de calor
sns.heatmap(corr_matrix_df, 
            mask=mask, 
            cmap=cmap, 
            vmin=-1.0, vmax=1.0,       # El rango de correlación de Pearson es siempre -1 a 1
            center=0,                  # Centramos el color en 0 (sin correlación)
            square=True,               # Forzamos que las celdas sean cuadradas
            linewidths=0.5,            # Líneas de separación finas entre celdas
            cbar_kws={"shrink": 0.8},  # Ajustamos el tamaño de la barra de leyenda
            annot=True,                # Mostramos los valores numéricos
            fmt=".2f",                 # Redondeamos a 2 decimales
            annot_kws={"size": 8})     # Reducimos un poco el texto para que quepa bien

# 9. Ajustes finales de diseño
plt.title("Matriz de Correlación de Vuelos", fontsize=20, pad=20)
plt.xticks(rotation=45, ha='right')    # Inclinamos las etiquetas del eje X para que no se superpongan
plt.yticks(rotation=0)
plt.tight_layout()

# Mostrar la gráfica
plt.show()

IllegalArgumentException: DELAY does not exist. Available: YEAR, MONTH, DAY, DAY_OF_WEEK, AIRLINE, FLIGHT_NUMBER, TAIL_NUMBER, ORIGIN_AIRPORT, DESTINATION_AIRPORT, SCHEDULED_DEPARTURE, DEPARTURE_TIME, DEPARTURE_DELAY, TAXI_OUT, WHEELS_OFF, SCHEDULED_TIME, ELAPSED_TIME, AIR_TIME, DISTANCE, WHEELS_ON, TAXI_IN, SCHEDULED_ARRIVAL, ARRIVAL_TIME, ARRIVAL_DELAY, DIVERTED, CANCELLED, CANCELLATION_REASON, AIR_SYSTEM_DELAY, SECURITY_DELAY, AIRLINE_DELAY, LATE_AIRCRAFT_DELAY, WEATHER_DELAY, AIRLINE, ORIGIN_LAT, ORIGIN_LONG